# 05 — Transforming
**Goal:** Create new columns, reshape data, and extract information from dates and strings.

Topics:
- `assign()` — add columns cleanly
- `apply()` — apply a function to rows or columns
- `map()` — map values using a dictionary
- `.str` accessor — string operations
- `.dt` accessor — date operations
- `cut()` / `qcut()` — bin numeric values
- `lambda` — anonymous functions used throughout pandas

In [12]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/funnel_data.csv', parse_dates=['date'])
print(df.shape)
df.head(3)

(450, 12)


,date,channel,visita_landing,inicio_solicitud,datos_personales,otp,datos_financieros,carga_documentos,evaluacion_crediticia,aprobacion,firma_digital,activacion_tarjeta
0,2024-01-01,organic,1010,388,291,207,165,99,89,48,39,27
1,2024-01-01,paid,1021,321,240,163,130,78,70,32,26,18
2,2024-01-01,email,481,202,151,102,89,53,47,25,20,14


## 1. `assign()` — add multiple columns at once
Cleaner than `df['col'] = ...` when adding several columns — and chainable.

In [13]:
df = df.assign(
    # Basic derived metrics
    cvr         = lambda x: x['activacion_tarjeta'] / x['visita_landing'] * 100,
    otp_rate    = lambda x: x['otp'] / x['inicio_solicitud'] * 100,
    doc_rate    = lambda x: x['carga_documentos'] / x['datos_personales'] * 100,
    approval_rate = lambda x: x['aprobacion'] / x['evaluacion_crediticia'] * 100,

    # Date parts — extracted from the date column
    year        = lambda x: x['date'].dt.year,
    month       = lambda x: x['date'].dt.month,
    week        = lambda x: x['date'].dt.isocalendar().week.astype(int),
    weekday     = lambda x: x['date'].dt.day_name(),
    is_weekend  = lambda x: x['date'].dt.dayofweek >= 5,
)

print(df[['date','channel','cvr','otp_rate','weekday','is_weekend']].head(8))

        date  channel       cvr   otp_rate  weekday  is_weekend
0 2024-01-01  organic  2.673267  53.350515   Monday       False
1 2024-01-01     paid  1.762977  50.778816   Monday       False
2 2024-01-01    email  2.910603  50.495050   Monday       False
3 2024-01-01   social  1.351351  50.485437   Monday       False
4 2024-01-01   direct  2.325581  50.000000   Monday       False
5 2024-01-02  organic  2.651113  53.443526  Tuesday       False
6 2024-01-02     paid  1.790281  50.813008  Tuesday       False
7 2024-01-02    email  2.800000  50.476190  Tuesday       False


## 2. `.dt` accessor — extract date components

In [14]:
# .dt gives you access to all datetime properties
print('Date parts available via .dt:')
print('year:       ', df['date'].dt.year.unique())
print('month:      ', df['date'].dt.month.unique())
print('day:        ', df['date'].dt.day[:5].tolist())
print('day_name:   ', df['date'].dt.day_name()[:3].tolist())
print('dayofweek:  ', df['date'].dt.dayofweek[:5].tolist())  # 0=Mon, 6=Sun
print('quarter:    ', df['date'].dt.quarter.unique())
print('week:       ', df['date'].dt.isocalendar().week[:3].tolist())

Date parts available via .dt:
year:        [2024]
month:       [1 2 3]
day:         [1, 1, 1, 1, 1]
day_name:    ['Monday', 'Monday', 'Monday']
dayofweek:   [0, 0, 0, 0, 0]
quarter:     [1]
week:        [1, 1, 1]


In [15]:
# Practical: label weeks and months for groupby
df['month_label'] = df['date'].dt.to_period('M').astype(str)   # '2024-01'
df['week_label']  = df['date'].dt.to_period('W').astype(str)   # '2024-01-01/2024-01-07'

print(df[['date','month_label','week_label']].drop_duplicates().head(8))

         date month_label             week_label
0  2024-01-01     2024-01  2024-01-01/2024-01-07
5  2024-01-02     2024-01  2024-01-01/2024-01-07
10 2024-01-03     2024-01  2024-01-01/2024-01-07
15 2024-01-04     2024-01  2024-01-01/2024-01-07
20 2024-01-05     2024-01  2024-01-01/2024-01-07
25 2024-01-06     2024-01  2024-01-01/2024-01-07
30 2024-01-07     2024-01  2024-01-01/2024-01-07
35 2024-01-08     2024-01  2024-01-08/2024-01-14


## 3. `.str` accessor — string operations on columns

In [16]:
# Simulate a messy campaign column
df_paid = df[df['channel'] == 'paid'].copy()
campaigns = ['BRAND_Search_CL', 'non_brand_SEARCH_CL', 'Remarketing-CL',
             'brand_Display_CL', 'NON_BRAND_display_cl']
df_paid['campaign_raw'] = (campaigns * (len(df_paid) // len(campaigns) + 1))[:len(df_paid)]

# Clean and extract info from the campaign name
df_paid = df_paid.assign(
    campaign    = lambda x: x['campaign_raw'].str.lower().str.strip(),
    is_brand    = lambda x: x['campaign_raw'].str.lower().str.contains('brand'),
    ad_type     = lambda x: x['campaign_raw'].str.lower()
                              .str.extract(r'(search|display|remarketing)')[0],
    country     = lambda x: x['campaign_raw'].str.extract(r'_(\w{2})$')[0].str.upper(),
)

print(df_paid[['campaign_raw','campaign','is_brand','ad_type','country']].drop_duplicates())

            campaign_raw              campaign  is_brand      ad_type country
1        BRAND_Search_CL       brand_search_cl      True       search      CL
6    non_brand_SEARCH_CL   non_brand_search_cl      True       search      CL
11        Remarketing-CL        remarketing-cl     False  remarketing     NaN
16      brand_Display_CL      brand_display_cl      True      display      CL
21  NON_BRAND_display_cl  non_brand_display_cl      True      display      CL


In [17]:
# Common .str methods:
s = pd.Series(['  Organic  ', 'PAID', 'Email_CL', 'social-mx'])

print('strip:   ', s.str.strip().tolist())
print('lower:   ', s.str.lower().tolist())
print('upper:   ', s.str.upper().tolist())
print('replace: ', s.str.replace('_','-').tolist())
print('split:   ', s.str.split('-').tolist())
print('contains:', s.str.lower().str.contains('organic').tolist())
print('len:     ', s.str.len().tolist())

strip:    ['Organic', 'PAID', 'Email_CL', 'social-mx']
lower:    ['  organic  ', 'paid', 'email_cl', 'social-mx']
upper:    ['  ORGANIC  ', 'PAID', 'EMAIL_CL', 'SOCIAL-MX']
replace:  ['  Organic  ', 'PAID', 'Email-CL', 'social-mx']
split:    [['  Organic  '], ['PAID'], ['Email_CL'], ['social', 'mx']]
contains: [True, False, False, False]
len:      [11, 4, 8, 9]


## 4. `map()` — replace values using a dictionary

In [18]:
# Map channel names to Spanish labels for a report
channel_labels = {
    'organic': 'Orgánico',
    'paid':    'Pagado',
    'email':   'Email',
    'social':  'Social',
    'direct':  'Directo',
}

df['channel_es'] = df['channel'].map(channel_labels)
print(df[['channel','channel_es']].drop_duplicates())

   channel channel_es
0  organic   Orgánico
1     paid     Pagado
2    email      Email
3   social     Social
4   direct    Directo


## 5. `apply()` — apply any function row by row or column by column

In [19]:
# apply() on a column (axis=0 or just Series.apply)
# Classify CVR into performance tiers
def cvr_tier(cvr):
    if cvr >= 3.0:  return 'High'
    if cvr >= 1.5:  return 'Medium'
    return 'Low'

df['cvr_tier'] = df['cvr'].apply(cvr_tier)
print(df['cvr_tier'].value_counts())

# Same thing with np.select (faster for large data)
conditions = [df['cvr'] >= 3.0, df['cvr'] >= 1.5]
choices    = ['High', 'Medium']
df['cvr_tier'] = np.select(conditions, choices, default='Low')
print(df['cvr_tier'].value_counts())

cvr_tier
Medium    370
Low        73
High        7
Name: count, dtype: int64
cvr_tier
Medium    370
Low        73
High        7
Name: count, dtype: int64


In [20]:
# apply() across multiple columns (axis=1 = row by row)
# Use sparingly — it's slow on large DataFrames
def funnel_health(row):
    """Return a health score based on multiple metrics."""
    score = 0
    if row['cvr'] > 2.0:      score += 1
    if row['otp_rate'] > 65:  score += 1
    if row['doc_rate'] > 55:  score += 1
    return score

df['health_score'] = df.apply(funnel_health, axis=1)
print(df['health_score'].value_counts().sort_index())

health_score
0    182
1    268
Name: count, dtype: int64


## 6. `cut()` / `qcut()` — bin numeric columns into categories

In [21]:
# cut() — fixed bin edges you define
df['cvr_band'] = pd.cut(
    df['cvr'],
    bins=[0, 1.0, 2.0, 3.0, float('inf')],
    labels=['<1%', '1-2%', '2-3%', '>3%']
)
print('CVR bands (fixed edges):'
)
print(df['cvr_band'].value_counts().sort_index())

print()

# qcut() — equal-sized quantile bins
df['visits_quartile'] = pd.qcut(
    df['visita_landing'],
    q=4,
    labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)']
)
print('Sessions quartiles (equal size):')
print(df['visits_quartile'].value_counts().sort_index())

CVR bands (fixed edges):
cvr_band
<1%       0
1-2%    182
2-3%    261
>3%       7
Name: count, dtype: int64

Sessions quartiles (equal size):
visits_quartile
Q1 (low)     113
Q2           112
Q3           112
Q4 (high)    113
Name: count, dtype: int64


## 7. `lambda` — anonymous functions in pandas

A `lambda` is a one-line function with no name. In pandas you use them constantly to write transformations inline without defining a separate `def`.

**Syntax:**
```python
lambda arguments: expression
```
It's equivalent to:
```python
def _(arguments):
    return expression
```

**Why `x` inside `assign()`?**  
When you pass `lambda x: ...` to `assign()`, pandas calls it with the **entire DataFrame** as `x`. This lets you reference multiple columns in a single expression — and even reference a column you just created in the same `assign()` call.

In [22]:
# ── 1. Lambda as a plain one-liner ───────────────────────────────────────────
double = lambda x: x * 2
print(double(5))          # 10

square_root = lambda x: x ** 0.5
print(square_root(9))     # 3.0

# With multiple arguments
rate = lambda num, den: num / den * 100
print(rate(27, 1010))     # 2.67...

10
3.0
2.6732673267326734


In [23]:
# ── 2. Lambda inside assign() — x is the whole DataFrame ─────────────────────
# You can chain columns that depend on each other in one assign() call
df2 = df[['date', 'channel', 'visita_landing', 'inicio_solicitud',
          'otp', 'activacion_tarjeta']].copy()

df2 = df2.assign(
    cvr      = lambda x: x['activacion_tarjeta'] / x['visita_landing'] * 100,
    otp_rate = lambda x: x['otp'] / x['inicio_solicitud'] * 100,
    # cvr_x2 references 'cvr' created just above — works because assign()
    # evaluates each lambda against the accumulating DataFrame
    cvr_x2   = lambda x: x['cvr'] * 2,
)

print(df2[['channel', 'cvr', 'otp_rate', 'cvr_x2']].head())

   channel       cvr   otp_rate    cvr_x2
0  organic  2.673267  53.350515  5.346535
1     paid  1.762977  50.778816  3.525955
2    email  2.910603  50.495050  5.821206
3   social  1.351351  50.485437  2.702703
4   direct  2.325581  50.000000  4.651163


In [24]:
# ── 3. Lambda inside apply() — x is a Series (one column) or a row ───────────

# On a Series: x is a scalar value
df['cvr_label'] = df['cvr'].apply(lambda x: f'{x:.1f}%')
print(df['cvr_label'].head(3))

# On a row (axis=1): x is a Series of all column values in that row
df['start_rate'] = df.apply(
    lambda x: x['inicio_solicitud'] / x['visita_landing'] * 100, axis=1
)
print(df[['channel', 'visita_landing', 'inicio_solicitud', 'start_rate']].head(3))

0    2.7%
1    1.8%
2    2.9%
Name: cvr_label, dtype: str
   channel  visita_landing  inicio_solicitud  start_rate
0  organic            1010               388   38.415842
1     paid            1021               321   31.439765
2    email             481               202   41.995842


In [25]:
# ── 4. Lambda inside groupby().apply() — x is the sub-DataFrame per group ────

# x here is the slice of df for one channel
channel_summary = df.groupby('channel').apply(
    lambda x: pd.Series({
        'days':         x['date'].nunique(),
        'total_visits': x['visita_landing'].sum(),
        'mean_cvr':     x['activacion_tarjeta'].sum() / x['visita_landing'].sum() * 100,
    })
)
print(channel_summary)

         days  total_visits  mean_cvr
channel                              
direct   90.0       22224.0  2.294816
email    90.0       42563.0  2.880436
organic  90.0       99551.0  2.679029
paid     90.0       86267.0  1.701694
social   90.0       34169.0  1.363809


In [26]:
# ── 5. Lambda inside sort_values() / map() / filter() ────────────────────────

# sort_values(key=) — apply a transform to the sort key without adding a column
df_sorted = df.sort_values(by='visita_landing', key=lambda s: -s)
print('Top 3 by visits:\n', df_sorted[['date', 'channel', 'visita_landing']].head(3))

# map() with a lambda — like apply() on a Series but more concise
df['channel_upper'] = df['channel'].map(lambda x: x.upper())
print('\nchannel_upper:', df['channel_upper'].unique())

# groupby().filter() — keep only groups matching a condition
# x is the sub-DataFrame for each group; must return a single bool
high_volume = df.groupby('channel').filter(lambda x: x['visita_landing'].sum() > 50_000)
print('\nChannels with >50k total visits:', high_volume['channel'].unique())

Top 3 by visits:
           date  channel  visita_landing
390 2024-03-19  organic            1413
395 2024-03-20  organic            1393
405 2024-03-22  organic            1386

channel_upper: <ArrowStringArray>
['ORGANIC', 'PAID', 'EMAIL', 'SOCIAL', 'DIRECT']
Length: 5, dtype: str

Channels with >50k total visits: <ArrowStringArray>
['organic', 'paid']
Length: 2, dtype: str


### When to use `lambda` vs `def`

| Situation | Use |
|---|---|
| One-liner, used once, inside `assign` / `apply` / `map` | `lambda` |
| Logic longer than one expression | `def` |
| Conditional with more than 2 branches | `def` (or `np.select`) |
| Reused in multiple places | `def` |

**Performance note:** `lambda` inside `apply(axis=1)` iterates row by row and is slow on large DataFrames. Prefer vectorized operations (`assign`, `np.where`, `np.select`) whenever possible.

## Summary
| Tool | Use when |
|---|---|
| `df.assign(col=lambda x: ...)` | Add one or more derived columns cleanly |
| `.dt.month` / `.dt.day_name()` | Extract date components |
| `.dt.to_period('M')` | Month/week labels for groupby |
| `.str.lower()` / `.str.strip()` | Normalize string columns |
| `.str.contains('x')` | Boolean flag from string pattern |
| `.str.extract(r'pattern')` | Extract substring with regex |
| `.map({old: new})` | Replace values from a dictionary |
| `.apply(func)` | Apply custom function to a column |
| `np.select(conditions, choices)` | Faster alternative to apply for tiers |
| `pd.cut(col, bins=[...])` | Bin into fixed ranges |
| `pd.qcut(col, q=4)` | Bin into equal-sized quartiles |
| `lambda x: expr` | Inline one-liner inside `assign` / `apply` / `map` / `groupby` |

**Next:** `06_groupby_aggregation.ipynb` — the most powerful tool in pandas for analytics.